📒 Notebook 02 — Train Word2Vec Models
🔹 Cell 1 — Install Dependencies

In [19]:
# ============================================
# Cell 1: Install Required Libraries
# ============================================

!pip install -q gensim==4.3.2
!pip install -q scipy==1.12.0
!pip install -q pandas
!pip install -q numpy
!pip install -q matplotlib
!pip install -q scikit-learn
!pip install -q tqdm
!pip install -q chardet

In [20]:
!pip install -q gensim

In [21]:
# ============================================
# Cell 2: Import Libraries
# ============================================

import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from gensim.models import Word2Vec
from gensim.models import KeyedVectors

warnings.filterwarnings("ignore")

print("✅ All libraries loaded successfully.")

✅ All libraries loaded successfully.


In [22]:
# ============================================
# Cell 3: Reproducibility
# ============================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Random Seed =", SEED)

Random Seed = 42


In [23]:
# ============================================
# Cell 4: File Paths
# ============================================

SHAHMUKHI_FILE = "/content/punjabi_shahmukhi_corpus_final.txt"
GURMUKHI_FILE = "/content/punjabi_gurmukhi_corpus_cleaned.txt"

MODELS_DIR = "/content/models"
RESULTS_DIR = "/content/results"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Files Ready")
print(SHAHMUKHI_FILE)
print(GURMUKHI_FILE)

Files Ready
/content/punjabi_shahmukhi_corpus_final.txt
/content/punjabi_gurmukhi_corpus_cleaned.txt


In [24]:
# ============================================
# Cell 5: Verify Files
# ============================================

for file in [SHAHMUKHI_FILE, GURMUKHI_FILE]:

    print("="*60)
    print(file)

    print("Exists:", os.path.exists(file))
    print("Size (MB):", round(os.path.getsize(file)/1024/1024,2))

    with open(file,"rb") as f:
        first=f.read(20)

    print(first)

/content/punjabi_shahmukhi_corpus_final.txt
Exists: True
Size (MB): 4.0
b'\xd9\x85\xda\xba\xd8\xa8\xdb\x8c \xd9\x85\xda\xba\xd8\xa8\xdb\x8c \xd8\xaa'
/content/punjabi_gurmukhi_corpus_cleaned.txt
Exists: True
Size (MB): 4.0
b'\xe0\xa8\xae\xe0\xa9\x81\xe0\xa9\xb0\xe0\xa8\xac\xe0\xa8\x88 \xe0\xa8\xae\xe0'


In [25]:
# ============================================
# Cell 6: Safe Streaming Corpus Loader
# ============================================

from gensim.models.word2vec import LineSentence


def validate_utf8_file(file_path):
    """
    Validate the file as UTF-8.
    If the only problem is an incomplete final character,
    create a repaired copy by removing at most 3 trailing bytes.
    """
    repaired_path = file_path.replace(".txt", "_repaired.txt")

    with open(file_path, "rb") as file:
        raw = file.read()

    for remove_n in range(4):
        candidate = raw if remove_n == 0 else raw[:-remove_n]

        try:
            candidate.decode("utf-8")

            if remove_n == 0:
                print(f"✅ UTF-8 valid: {file_path}")
                return file_path

            with open(repaired_path, "wb") as output:
                output.write(candidate)

            print(
                f"✅ Repaired {file_path} "
                f"by removing {remove_n} incomplete byte(s)."
            )

            return repaired_path

        except UnicodeDecodeError:
            continue

    raise UnicodeError(
        f"Encoding problem is not limited to the end of: {file_path}"
    )


SHAHMUKHI_VALID_FILE = validate_utf8_file(SHAHMUKHI_FILE)
GURMUKHI_VALID_FILE = validate_utf8_file(GURMUKHI_FILE)

print("\nFiles selected for training:")
print("Shahmukhi:", SHAHMUKHI_VALID_FILE)
print("Gurmukhi:", GURMUKHI_VALID_FILE)

✅ UTF-8 valid: /content/punjabi_shahmukhi_corpus_final.txt
✅ UTF-8 valid: /content/punjabi_gurmukhi_corpus_cleaned.txt

Files selected for training:
Shahmukhi: /content/punjabi_shahmukhi_corpus_final.txt
Gurmukhi: /content/punjabi_gurmukhi_corpus_cleaned.txt


In [26]:
# ============================================
# Cell 7: Streaming Sentence Sources
# ============================================

shahmukhi_sentences = LineSentence(SHAHMUKHI_VALID_FILE)
gurmukhi_sentences = LineSentence(GURMUKHI_VALID_FILE)

print("✅ Streaming corpus readers created.")

✅ Streaming corpus readers created.


In [27]:
# ============================================
# Cell 8: Preview Tokenized Segments
# ============================================

from itertools import islice

print("Shahmukhi examples:")
for sentence in islice(
    LineSentence(SHAHMUKHI_VALID_FILE),
    3
):
    print(sentence[:20])

print("\nGurmukhi examples:")
for sentence in islice(
    LineSentence(GURMUKHI_VALID_FILE),
    3
):
    print(sentence[:20])

Shahmukhi examples:
['مںبی', 'مںبی', 'تک', 'بںبی', 'بھارت', 'دے', 'مہاںراسٹر', 'سوبے', 'دی', 'راجدھانی', 'ہے', 'اتے', 'اہ', 'بھارت', 'دا', 'دوسرا', 'وڈا', 'مہاںنگر', 'ہے', 'بھارت']
['گربانی', 'سکھ', 'گروآں', 'دیآں', 'رچناواں', 'نوں', 'کہا', 'جاںدا', 'ہے', 'گربانی', 'سبد', 'دو', 'سبداں', 'توں', 'بنآ', 'ہے', 'گر', 'اتے', 'بانی', 'گر']
['جپ', 'جی', 'ساہب', 'جاں', 'جپ', 'جی', 'گرو', 'نانک', 'دےو', 'دی', 'لکھی', 'بانی', 'ہے', 'جو', 'گرو', 'گرںتھ', 'ساہب', 'وچ', 'سبھ', 'توں']

Gurmukhi examples:
['ਮੁੰਬਈ', 'ਮੁੰਬਈ', 'ਤੱਕ', 'ਬੰਬਈ', 'ਭਾਰਤ', 'ਦੇ', 'ਮਹਾਂਰਾਸ਼ਟਰ', 'ਸੂਬੇ', 'ਦੀ', 'ਰਾਜਧਾਨੀ', 'ਹੈ', 'ਅਤੇ', 'ਇਹ', 'ਭਾਰਤ', 'ਦਾ', 'ਦੂਸਰਾ', 'ਵੱਡਾ', 'ਮਹਾਂਨਗਰ', 'ਹੈ', 'ਭਾਰਤ']
['ਗੁਰਬਾਣੀ', 'ਸਿੱਖ', 'ਗੁਰੂਆਂ', 'ਦੀਆਂ', 'ਰਚਨਾਵਾਂ', 'ਨੂੰ', 'ਕਿਹਾ', 'ਜਾਂਦਾ', 'ਹੈ', 'ਗੁਰਬਾਣੀ', 'ਸ਼ਬਦ', 'ਦੋ', 'ਸ਼ਬਦਾਂ', 'ਤੋਂ', 'ਬਣਿਆ', 'ਹੈ', 'ਗੁਰ', 'ਅਤੇ', 'ਬਾਣੀ', 'ਗੁਰ']
['ਜਪੁ', 'ਜੀ', 'ਸਾਹਿਬ', 'ਜਾਂ', 'ਜਪੁ', 'ਜੀ', 'ਗੁਰੂ', 'ਨਾਨਕ', 'ਦੇਵ', 'ਦੀ', 'ਲਿਖੀ', 'ਬਾਣੀ', 'ਹੈ', 'ਜੋ', 'ਗੁਰੂ', 'ਗ੍ਰੰਥ', 'ਸਾਹਿਬ', 'ਵਿੱਚ', 'ਸਭ', 'ਤੋਂ']


In [28]:
# ============================================
# Cell 9: Word2Vec Configuration
# ============================================

VECTOR_SIZE = 200
WINDOW_SIZE = 5
MIN_COUNT = 5
EPOCHS = 15
WORKERS = 2
NEGATIVE_SAMPLES = 10

print("Word2Vec configuration")
print("Vector size:", VECTOR_SIZE)
print("Window size:", WINDOW_SIZE)
print("Minimum count:", MIN_COUNT)
print("Epochs:", EPOCHS)
print("Workers:", WORKERS)
print("Negative samples:", NEGATIVE_SAMPLES)

Word2Vec configuration
Vector size: 200
Window size: 5
Minimum count: 5
Epochs: 15
Workers: 2
Negative samples: 10


In [29]:
# ============================================
# Cell 10: Train Gurmukhi CBOW
# ============================================

gurmukhi_cbow = Word2Vec(
    sentences=LineSentence(GURMUKHI_VALID_FILE),
    vector_size=VECTOR_SIZE,
    window=WINDOW_SIZE,
    min_count=MIN_COUNT,
    workers=WORKERS,
    sg=0,  # 0 = CBOW
    negative=NEGATIVE_SAMPLES,
    epochs=EPOCHS,
    seed=SEED
)

print("✅ Gurmukhi CBOW training completed.")
print("Vocabulary size:", len(gurmukhi_cbow.wv))

✅ Gurmukhi CBOW training completed.
Vocabulary size: 6349


In [30]:
# ============================================
# Cell 11: Save Gurmukhi CBOW
# ============================================

GURMUKHI_CBOW_PATH = (
    f"{MODELS_DIR}/gurmukhi_cbow_word2vec.model"
)

gurmukhi_cbow.save(GURMUKHI_CBOW_PATH)

print("✅ Model saved to:")
print(GURMUKHI_CBOW_PATH)

✅ Model saved to:
/content/models/gurmukhi_cbow_word2vec.model


In [31]:
# ============================================
# Cell 12: Test Gurmukhi CBOW
# ============================================

query_word = "ਪੰਜਾਬ"

if query_word in gurmukhi_cbow.wv:
    neighbors = gurmukhi_cbow.wv.most_similar(
        query_word,
        topn=10
    )

    print("Nearest neighbors for:", query_word)

    for word, score in neighbors:
        print(word, round(score, 4))

else:
    print(
        query_word,
        "is not present in the model vocabulary."
    )

Nearest neighbors for: ਪੰਜਾਬ
ਪਾਕਿਸਤਾਨ 0.7285
ਹਰਿਆਣਾ 0.6624
ਇੱਥੋਂ 0.5911
ਸੂਬੇ 0.5748
ਬਠਿੰਡੇ 0.5701
ਹਿਮਾਚਲ 0.5667
ਰਾਜਸਥਾਨ 0.5653
ਜ਼ਿਲ੍ਹਿਆਂ 0.5651
ਦਫ਼ਤਰਾਂ 0.5609
ਮਾਲਵਾ 0.5578


In [32]:
# ============================================
# Cell 13: Train Word2Vec Function
# ============================================

def train_word2vec(
    corpus_file,
    model_name,
    architecture="cbow"
):

    sg = 0 if architecture.lower()=="cbow" else 1

    print(f"\nTraining {model_name} ({architecture})...")

    model = Word2Vec(
        sentences=LineSentence(corpus_file),
        vector_size=200,
        window=5,
        min_count=5,
        workers=2,
        sg=sg,
        negative=10,
        epochs=15,
        seed=42
    )

    save_path=f"{MODELS_DIR}/{model_name}.model"

    model.save(save_path)

    print("Saved:",save_path)
    print("Vocabulary:",len(model.wv))

    return model

In [15]:
gurmukhi_cbow = train_word2vec(
    GURMUKHI_VALID_FILE,
    "gurmukhi_cbow",
    "cbow"
)

gurmukhi_skipgram = train_word2vec(
    GURMUKHI_VALID_FILE,
    "gurmukhi_skipgram",
    "skipgram"
)

shahmukhi_cbow = train_word2vec(
    SHAHMUKHI_VALID_FILE,
    "shahmukhi_cbow",
    "cbow"
)

shahmukhi_skipgram = train_word2vec(
    SHAHMUKHI_VALID_FILE,
    "shahmukhi_skipgram",
    "skipgram"
)


Training gurmukhi_cbow (cbow)...
Saved: /content/models/gurmukhi_cbow.model
Vocabulary: 5205

Training gurmukhi_skipgram (skipgram)...
Saved: /content/models/gurmukhi_skipgram.model
Vocabulary: 5205

Training shahmukhi_cbow (cbow)...
Saved: /content/models/shahmukhi_cbow.model
Vocabulary: 6748

Training shahmukhi_skipgram (skipgram)...


Saved: /content/models/shahmukhi_skipgram.model
Vocabulary: 6748


In [16]:
# ============================================
# Cell 15: Training Summary
# ============================================

training_summary = pd.DataFrame({
    "Corpus": [
        "Punjabi Gurmukhi",
        "Punjabi Gurmukhi",
        "Punjabi Shahmukhi",
        "Punjabi Shahmukhi"
    ],
    "Architecture": [
        "CBOW",
        "Skip-Gram",
        "CBOW",
        "Skip-Gram"
    ],
    "Vector Size": [200, 200, 200, 200],
    "Window": [5, 5, 5, 5],
    "Min Count": [5, 5, 5, 5],
    "Epochs": [15, 15, 15, 15],
    "Embedding Vocabulary": [
        len(gurmukhi_cbow.wv),
        len(gurmukhi_skipgram.wv),
        len(shahmukhi_cbow.wv),
        len(shahmukhi_skipgram.wv)
    ]
})

training_summary

,Corpus,Architecture,Vector Size,Window,Min Count,Epochs,Embedding Vocabulary
0,Punjabi Gurmukhi,CBOW,200,5,5,15,5205
1,Punjabi Gurmukhi,Skip-Gram,200,5,5,15,5205
2,Punjabi Shahmukhi,CBOW,200,5,5,15,6748
3,Punjabi Shahmukhi,Skip-Gram,200,5,5,15,6748


In [17]:
training_summary.to_csv(
    f"{RESULTS_DIR}/training_summary.csv",
    index=False
)

print("Training summary saved.")

Training summary saved.


In [18]:
# ============================================
# Export Standalone Embedding Vectors
# ============================================

EXPORT_DIR = "/content/exported_vectors"

os.makedirs(EXPORT_DIR, exist_ok=True)

gurmukhi_cbow.wv.save(
    f"{EXPORT_DIR}/gurmukhi_cbow.kv",
    separately=[]
)

gurmukhi_skipgram.wv.save(
    f"{EXPORT_DIR}/gurmukhi_skipgram.kv",
    separately=[]
)

shahmukhi_cbow.wv.save(
    f"{EXPORT_DIR}/shahmukhi_cbow.kv",
    separately=[]
)

shahmukhi_skipgram.wv.save(
    f"{EXPORT_DIR}/shahmukhi_skipgram.kv",
    separately=[]
)

print("✅ Four standalone vector files saved.")

✅ Four standalone vector files saved.


In [33]:
# Save all four models again
import os
import shutil

MODELS_DIR = "/content/models"
os.makedirs(MODELS_DIR, exist_ok=True)

gurmukhi_cbow.save(
    f"{MODELS_DIR}/gurmukhi_cbow.model"
)

gurmukhi_skipgram.save(
    f"{MODELS_DIR}/gurmukhi_skipgram.model"
)

shahmukhi_cbow.save(
    f"{MODELS_DIR}/shahmukhi_cbow.model"
)

shahmukhi_skipgram.save(
    f"{MODELS_DIR}/shahmukhi_skipgram.model"
)

# Create one ZIP file
shutil.make_archive(
    "/content/punjabi_word2vec_models",
    "zip",
    MODELS_DIR
)

print("✅ ZIP created:")
print("/content/punjabi_word2vec_models.zip")

✅ ZIP created:
/content/punjabi_word2vec_models.zip
